# 00 — Pipeline Lengkap: A1 → A2 → A3

Notebook orkestrasi yang menjalankan **seluruh rantai** penelitian dalam satu
tempat:

```
Citra mentah
  → [A1] praproses (grayscale, resize 128x128, equalization)   → *_preprocessed.npz
  → [A2] SWT Haar A1 + PHOG (680-D)                            → *_features.npz
  → [A3] LDA + classifier (baseline SVM-RBF, LS-SVM OvO)       → akurasi
  → [A1] uji noise & evaluasi subject-independent
```

## Cara pakai

Atur `USE_CACHE` pada sel berikutnya:

| Nilai | Perilaku | Perkiraan waktu |
|---|---|---|
| `True` | lompati langkah berat, pakai artefak `.npz` yang sudah ada | ~1 menit |
| `False` | hitung ulang dari citra mentah (bukti lengkap) | ~40–70 menit |

Notebook ini **memanggil modul** di `research/` — tidak menyalin logika.
Dengan begitu, perbaikan pada modul otomatis ikut terpakai di sini.

## Troubleshooting

| Gejala | Penyebab | Solusi |
|---|---|---|
| `FileNotFoundError: File fitur ... tidak ditemukan` | artefak A2 belum dibuat | jalankan notebook `02-a2-feature-extraction` dulu |
| `ModuleNotFoundError: features` | PyWavelets diblokir | `USE_CACHE = True`, atau jalankan `\run.ps1 check` |
| `UnicodeEncodeError: charmap` | console Windows cp1252 | sudah ditangani `run.ps1` (PYTHONIOENCODING=utf-8); saat manual: `$env:PYTHONIOENCODING="utf-8"` |

Seluruh path pada `config.yaml` bersifat **relatif terhadap root repo** (lokasi
`config.yaml`), bukan terhadap CWD notebook. Jadi notebook ini aman dijalankan
dari `research/pipeline/`.


## 1. Setup & pilihan mode

In [9]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def _find_repo_root(start: Path) -> Path:
    """Telusuri ke atas sampai menemukan config.yaml (tahan dipindah folder)."""
    for candidate in [start, *start.parents]:
        if (candidate / "config.yaml").exists():
            return candidate
    raise RuntimeError(f"config.yaml tidak ditemukan dari {start}")


repo_root = _find_repo_root(Path.cwd().resolve())

# Modul .py berada di research/<divisi>/
for sub in ("a1-data-preprocessing", "a2-feature-engineering", "a3-model-optimization"):
    p = str(repo_root / "research" / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

from baseline import load_config, resolve_path

config = load_config(repo_root / "config.yaml")
print("repo_root :", repo_root)
print("proyek    :", config["project"]["name"])
print("seed      :", config["project"]["random_seed"])

USE_CACHE = True   # ubah ke False untuk menghitung ulang dari awal
print('USE_CACHE =', USE_CACHE)

repo_root : D:\Semester5\pcvk\tugas_akhir
proyek    : TA-PCVK
seed      : 42
USE_CACHE = True


## 2. [A1] Dataset & pemeriksaan awal

In [10]:
meta = {}
for key in ("ck", "jaffe"):
    path = resolve_path(config, config["paths"][f"{key}_features"])
    d = np.load(path, allow_pickle=True)
    meta[key] = {
        "fitur": d["F"].shape,
        "kelas": d["class_names"].tolist(),
        "subjek": len(set(d["subjects"].astype(str).tolist())),
    }
    print(f"{key.upper():6s} fitur={d['F'].shape}  kelas={len(meta[key]['kelas'])}  subjek={meta[key]['subjek']}")

pd.DataFrame(meta).T

CK     fitur=(981, 680)  kelas=7  subjek=118
JAFFE  fitur=(213, 680)  kelas=7  subjek=10


,fitur,kelas,subjek
ck,"(981, 680)","[anger, contempt, disgust, fear, happy, sadnes...",118
jaffe,"(213, 680)","[anger, disgust, fear, happy, neutral, sadness...",10


## 3. [A2] Ekstraksi fitur SWT + PHOG

Dimensi mengikuti persamaan (2) jurnal:

$$F = K \times \sum_{l=0}^{L} 4^l = 8 \times (1+4+16+64) = 680$$

Jika `USE_CACHE = False`, sel ini mengekstrak ulang 1.194 citra (±5–6 menit).


In [11]:
if not USE_CACHE:
    # Impor lazy: modul ini membutuhkan PyWavelets, jadi hanya dimuat
    # ketika ekstraksi benar-benar dijalankan.
    from features import extract_features

    for key in ("ck", "jaffe"):
        src = np.load(resolve_path(config, config["paths"][f"{key}_preprocessed"]), allow_pickle=True)
        X = src["X"]
        print(f"[{key}] ekstraksi {len(X)} citra ...")
        F = extract_features(X)
        assert F.shape == (len(X), 680), F.shape
        dst = resolve_path(config, config["paths"][f"{key}_features"])
        np.savez_compressed(
            dst, F=F, y=src["y"], subjects=src["subjects"], class_names=src["class_names"]
        )
        print(f"  -> {dst.name} {F.shape}")
else:
    print("USE_CACHE=True: artefak fitur dipakai apa adanya.")

USE_CACHE=True: artefak fitur dipakai apa adanya.


## 4. [A3] Baseline LDA + SVM-RBF

In [12]:
from baseline import run_baseline_evaluation

hasil_baseline = {}
for key in ("ck", "jaffe"):
    res = run_baseline_evaluation(config["paths"][f"{key}_features"], config=config)
    hasil_baseline[key] = res
    print(f"{key.upper():6s} akurasi = {res['mean_accuracy']*100:.2f}%  (std {res['std_accuracy']*100:.2f})")

CK     akurasi = 97.66%  (std 1.00)
JAFFE  akurasi = 81.27%  (std 8.37)


## 5. [A3] LS-SVM One-vs-One + grid search

In [13]:
from evaluate_lssvm import run_lssvm_multiclass_evaluation

hasil_lssvm = {}
for key in ("ck", "jaffe"):
    res = run_lssvm_multiclass_evaluation(config["paths"][f"{key}_features"], config=config)
    hasil_lssvm[key] = res
    print(f"{key.upper():6s} akurasi = {res['mean_accuracy']*100:.2f}%  (std {res['std_accuracy']*100:.2f})")

  Fold 1/5 -> Best params (gamma=1.0, sigma=5.0) -> Acc: 95.43%
  Fold 2/5 -> Best params (gamma=1.0, sigma=5.0) -> Acc: 95.92%
  Fold 3/5 -> Best params (gamma=1.0, sigma=5.0) -> Acc: 97.96%
  Fold 4/5 -> Best params (gamma=1.0, sigma=5.0) -> Acc: 97.45%
  Fold 5/5 -> Best params (gamma=1.0, sigma=5.0) -> Acc: 96.43%
CK     akurasi = 96.64%  (std 0.94)
  Fold 1/5 -> Best params (gamma=1.0, sigma=5.0) -> Acc: 65.12%
  Fold 2/5 -> Best params (gamma=1.0, sigma=3.0) -> Acc: 86.05%
  Fold 3/5 -> Best params (gamma=1.0, sigma=3.0) -> Acc: 83.72%
  Fold 4/5 -> Best params (gamma=500.0, sigma=1.5) -> Acc: 83.33%
  Fold 5/5 -> Best params (gamma=1.0, sigma=5.0) -> Acc: 78.57%
JAFFE  akurasi = 79.36%  (std 7.52)


## 6. [A1] Artefak noise Gaussian

In [14]:
from noise import DEFAULT_VARIANCES, build_noise_artifacts

written = build_noise_artifacts(config, variances=DEFAULT_VARIANCES)
print(f"{len(written)} artefak noise tersedia di {config['paths']['a1_noise_dir']}/")

[noise] ck: 981 citra, 7 kelas
  [skip] ck_noise_v0.01.npz sudah ada
  [skip] ck_noise_v0.02.npz sudah ada
  [skip] ck_noise_v0.05.npz sudah ada
  [skip] ck_noise_v0.10.npz sudah ada
[noise] jaffe: 213 citra, 7 kelas
  [skip] jaffe_noise_v0.01.npz sudah ada
  [skip] jaffe_noise_v0.02.npz sudah ada
  [skip] jaffe_noise_v0.05.npz sudah ada
  [skip] jaffe_noise_v0.10.npz sudah ada
8 artefak noise tersedia di data/a1-noise/


## 7. [A1] Split acak vs subject-independent

In [15]:
from evaluate_subject_independent import compare_splits

hasil_split = {}
for key in ("ck", "jaffe"):
    hasil_split[key] = compare_splits(config["paths"][f"{key}_features"], config=config, verbose=False)
    print(f"{key.upper():6s} acak={hasil_split[key]['random']['mean_accuracy']*100:.2f}%  "
          f"subject={hasil_split[key]['subject']['mean_accuracy']*100:.2f}%")

CK     acak=96.64%  subject=48.21%
JAFFE  acak=80.31%  subject=38.22%


## 8. Ringkasan seluruh hasil

In [16]:
rows = []
for key in ("ck", "jaffe"):
    rows.append({"Dataset": key.upper(), "Metode": "LDA + SVM-RBF",
                 "Akurasi (%)": round(hasil_baseline[key]["mean_accuracy"] * 100, 2)})
    rows.append({"Dataset": key.upper(), "Metode": "LDA + OvO LS-SVM (grid search)",
                 "Akurasi (%)": round(hasil_lssvm[key]["mean_accuracy"] * 100, 2)})
    rows.append({"Dataset": key.upper(), "Metode": "OvO LS-SVM (split subject-independent)",
                 "Akurasi (%)": round(hasil_split[key]["subject"]["mean_accuracy"] * 100, 2)})

df_ringkas = pd.DataFrame(rows)
df_ringkas

,Dataset,Metode,Akurasi (%)
0,CK,LDA + SVM-RBF,97.66
1,CK,LDA + OvO LS-SVM (grid search),96.64
2,CK,OvO LS-SVM (split subject-independent),48.21
3,JAFFE,LDA + SVM-RBF,81.27
4,JAFFE,LDA + OvO LS-SVM (grid search),79.36
5,JAFFE,OvO LS-SVM (split subject-independent),38.22


## 9. Pembahasan

- **Baseline dan LS-SVM memberi akurasi sebanding** pada CK+, sedangkan pada
  JAFFE keduanya turun karena hanya ada 10 subjek (213 citra).
- **Angka split acak tidak boleh diklaim sebagai generalisasi.** Ketika subjek
  dipisah dengan benar, akurasi CK+ maupun JAFFE turun sangat tajam.
- **Dataset kita berbeda dari jurnal** (981 citra/7 kelas vs 450/8 kelas),
  sehingga angka jurnal hanya dipakai sebagai kolom pembanding, bukan target.

## 10. Rujukan

Kar, N.B., Nayak, D.R., Babu, K.S., & Zhang, Y.-D. (2021). *A hybrid feature
descriptor with Jaya optimised least squares SVM for facial expression
recognition.* IET Image Processing, 15(7), 1471–1483.
